In [1]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START , END

from langgraph.graph.message import add_messages
## add message is a reducer for appending messages to the state graph

In [2]:
class State(TypedDict):

    messages: Annotated[list,add_messages]

graph_builder=StateGraph(State)

In [3]:
from dotenv import load_dotenv
load_dotenv()



True

In [4]:
from langchain_groq import ChatGroq

model=ChatGroq(
    model="openai/gpt-oss-20b"
)

model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000287E124B810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000287E23768D0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### chatbot is like a node, and for every node we need a node definition 


In [5]:
## node functionality

def chatbot(state:State):
    return {"messages":[model.invoke(state["messages"])]}  ## state["messages"] this message is the user input, and the response will be appended in the messages variable(outer message in this code.)
    

In [6]:
## building graph

graph_builder= StateGraph(State)

## adding node

graph_builder.add_node("llmchatbot",chatbot) ## the name of the node, and the node definition

graph_builder.add_edge(START,"llmchatbot")

graph_builder.add_edge("llmchatbot",END)

# compile the graph
graph=graph_builder.compile()
 

In [7]:
response=graph.invoke({"messages":"Hi"})
response["messages"][-1].content

'Hello! How can I help you today?'

### chatbot with tool

In [8]:

from langchain_tavily import TavilySearch

tool=TavilySearch(max_results=2)
tool.invoke("what is langsmith?")

{'query': 'what is langsmith?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.langchain.com/blog/announcing-langsmith',
   'title': 'Announcing LangSmith, a unified platform for debugging, testing, evaluating, and monitoring your LLM applications',
   'content': 'Today, we’re introducing LangSmith, a platform to help developers close the gap between prototype and production. It’s designed for building and iterating on products that can harness the power–and wrangle the complexity–of LLMs.\n\nLangSmith is now in closed beta. So if you’re looking for a robust, unified, system for debugging, testing, evaluating, and monitoring your LLM applications, sign up here.\n\nHow did we get here? [...] This deep visibility into model performance has been particularly helpful for teams developing complex applications. LangSmith helped Streamlit and Snowflake implement agents that could intelligently and reliably answer questions about their data. [..

In [16]:
def divide(a:int,b:int)->int:

    """
    divide a with b
    """
    return a/b
    

In [11]:
tools=[tool,divide]

llm_with_tools=model.bind_tools(tools)


In [12]:
llm_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000287E124B810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000287E23768D0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'A search engine optimized for comprehensive, accurate, and 

In [ ]:
from langgraph.graph import StateGraph,START,END

from langgraph.prebuilt import tools_condition

from langgraph.prebuilt import ToolNode

#node definition

def tool_calling(state:State):

    return {"messages":[llm_with_tools.invoke(state["messages"])]}


## graph

builder =StateGraph(State)

builder.add_node("tool_calling_llm",tool_calling)

builder.add_node("tools",ToolNode(tools))

## add edges

builder.add_edge(START,"tool_calling_llm")

builder.add_conditional_edges(

    "tool_calling_llm",

    tools_condition
)

builder.add_edge("tools",END)

## compile the graph

graph=builder.compile() 








In [24]:
response=graph.invoke({"messages":"what is the recent indian news?"})
response['messages'][-1].content

'{"query": "recent Indian news", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://timesofindia.indiatimes.com/india/most-comprehensive-plan-ukraine-confirms-india-playing-role-in-mediating-energy-truce-in-black-sea/articleshow/134659507.cms", "title": "\'Most comprehensive plan\': Ukraine confirms India playing role in mediating energy truce in Black Sea | India News - The Times of India", "content": "India Vs China Womens HockeyGurinder SanghaParveen HoodaTirupati Factory BlastCJP ProtestAastha Kunj Gang RapePM ModiDelhi-Dehradun Expressway TheftSachin WazeGangotri National ParkDelhi MurderKolkata StabbingDhanraj NathwaniMeerut Blue Drum CaseMumbai Superfast TrainAbhijeet DipkeAsian Games Medal Tally Winner ListSmit MachchharsAdar PoonawalaDelhi MolestationIndias Factory GrowthSandhya ReddyAnthropicRBI Repo Rate HikeTN Dementia CasesWeather ForecastDelhi Airport Drone SeizedGulzar Singh [...] over \'cattle theft\' in DarjeelingRebel TMC MLA Bayro

In [26]:
response=graph.invoke({"messages":"what is 28 divided by 4?"})

response['messages'][-1].content

'28 ÷ 4 = **7**.'